In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import joblib

# churn_pipeline.py
# End-to-End ML pipeline for Telco Churn using scikit-learn Pipeline API
# Usage: adjust DATA_PATH and run as a script or import functions.


DATA_PATH = "Telco-Customer-Churn.csv"  # update path if needed
OUTPUT_PIPELINE = "churn_pipeline.joblib"
RANDOM_STATE = 42

def load_and_clean(path=DATA_PATH):
    df = pd.read_csv(path)
    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])
    # Convert TotalCharges to numeric (some datasets have spaces)
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    # Target
    y = df["Churn"].map({"Yes": 1, "No": 0})
    X = df.drop(columns=["Churn"])
    return X, y

def build_pipeline(X):
    # identify column types
    numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ])
    return pipe

def tune_and_train(X_train, y_train, pipeline):
    param_grid = [
        {
            "clf": [LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)],
            "clf__C": [0.01, 0.1, 1.0, 10.0],
            "clf__penalty": ["l2"],
            "clf__solver": ["lbfgs"]
        },
        {
            "clf": [RandomForestClassifier(random_state=RANDOM_STATE)],
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_split": [2, 5]
        }
    ]

    gs = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=1
    )
    gs.fit(X_train, y_train)
    return gs

def save_pipeline(estimator, path=OUTPUT_PIPELINE):
    joblib.dump(estimator, path, compress=3)
    return path

def main():
    X, y = load_and_clean(DATA_PATH)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    pipe = build_pipeline(X_train)
    gs = tune_and_train(X_train, y_train, pipe)

    best_pipe = gs.best_estimator_
    preds_proba = best_pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds_proba)

    model_path = save_pipeline(best_pipe, OUTPUT_PIPELINE)

    print("Best params:", gs.best_params_)
    print(f"Test ROC AUC: {auc:.4f}")
    print(f"Saved pipeline to: {model_path}")

if __name__ == "__main__":
    main()